# EMNIST Balanced Model Creation

This notebook documents the **model-development workflow** used to create and select the EMNIST Balanced reference model for subsequent FPGA and Unity implementation.

It intentionally stops **before FPGA integer-datapath design, Vivado project construction, camera integration, and Unity application development**. Those steps are documented in the respective implementation branches.

## Scope

This notebook covers:

1. EMNIST Balanced dataset verification and orientation
2. historical fully connected network experiments
3. the 784–200–47 dense-network experiment
4. the initial CNN reference experiment
5. the improved CNN experiment
6. the compact CNN selected for deployment
7. model validation and saved checkpoint evidence
8. preliminary software quantization comparison
9. selected artifacts, model provenance, and reproducibility notes

### Reference resources

- Official EMNIST dataset: <https://www.nist.gov/itl/products-and-services/emnist-dataset>
- Model-generation scripts: **`EMNIST-Models-Code`** branch
- Training logs, summaries, model checkpoints, figures, and this notebook: **`EMNIST-Models-Data-and-Notebook`** branch

The original EMNIST gzip dataset files are **not committed** to this branch. The retained output files document the runs already performed.

## Files expected in the same directory

The GitHub branch stores the notebook alongside the experiment outputs. Principal files used by the notebook include:

```text
EMNIST_Model_Development_Reproducibility.ipynb
EMNIST_Balanced_Verification_Summary.txt
EMNIST_Balanced_Orientation_Check.png
EMNIST_Balanced_Reference_Vectors.npz
EMNIST_Balanced_baseline_Training_Summary.json
EMNIST_Balanced_expanded_Training_Summary.json
training_summary_mlp200.json
training_summary.json
training_summary_improved.json
training_summary_compact.json
validation_history.csv
validation_history_improved.csv
validation_history_compact.csv
validation_history_mlp200.csv
test_confusion_matrix.csv
test_confusion_matrix_compact.csv
quantization_comparison.csv
best_cnn_checkpoint.pt
best_cnn_checkpoint_improved.pt
best_cnn_checkpoint_compact.pt
best_mlp_checkpoint.pt
final_mlp_checkpoint.pt
```

Additional historical images and text-based weight/bias exports are also retained in the branch. **The notebook uses the actual repository filenames** rather than inventing new experimental artifacts. Missing optional files are reported as unavailable instead of replaced by estimated results.

The training scripts themselves are maintained in the companion code branch.

## 1. Historical EMNIST training evidence

The original experiments began by adapting the earlier `network2`-style fully connected workflow to EMNIST Balanced, then evaluated expanded and 200-hidden-unit networks. CNN development followed because handwritten characters benefit from spatial feature extraction.

The images retained here provide **training provenance**, not a substitute for the validated results in the saved JSON summaries.

### Early training outputs

![Early EMNIST training, first screenshot](emnist_epoch_100_results1.png)

![Early EMNIST training, second screenshot](emnist_epoch_100_results2.png)

The historical 100-epoch screenshots belong to the earlier training workflow. They should not be interpreted as the final compact CNN evaluation.

## 2. Verify the EMNIST Balanced dataset

The official Balanced split consists of **112,800 training images** and **18,800 test images**, divided evenly among **47 classes**, each containing **28 × 28 grayscale pixels**.

The original EMNIST IDX image arrays require a transpose to display characters upright. The preparation scripts perform that correction before flattening or CNN input construction.

### Orientation verification

![EMNIST raw and corrected image orientation](EMNIST_Balanced_Orientation_Check.png)

The corrected orientation must be applied consistently in training, software inference, and subsequent deployment. Applying the correction twice would alter the inputs.

In [ ]:
from pathlib import Path
import json
import csv
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()

def artifact(name):
    path = ROOT / name
    if not path.is_file():
        print(f"Not found in this directory: {name}")
        return None
    return path

summary_file = artifact("EMNIST_Balanced_Verification_Summary.txt")
if summary_file:
    print(summary_file.read_text(encoding="utf-8"))

### Reference vectors

`EMNIST_Balanced_Reference_Vectors.npz` contains **one upright, unnormalized test image per class**, along with the corresponding class IDs and dataset indices. These vectors document the input orientation and can support later validation, but they do not constitute a trained-model accuracy measurement.

In [ ]:
ref_path = artifact("EMNIST_Balanced_Reference_Vectors.npz")
if ref_path:
    with np.load(ref_path) as vectors:
        for key in vectors.files:
            print(key, vectors[key].shape, vectors[key].dtype)
        assert vectors['pixels_u8'].shape == (47, 784)
        assert np.array_equal(vectors['class_ids'], np.arange(47))

## 3. Fully connected network experiments

The early EMNIST experiments extended the MNIST dense-network approach to **47 outputs**. The original training and expanded-network runs are preserved as historical experiments; they are not the selected model used for later FPGA deployment.

The corresponding source scripts are `train_emnist.py`, `train_emnist_expanded.py`, `network2_emnist.py`, and `emnist_loader.py` in the code branch.

In [ ]:
def load_json(name):
    p = artifact(name)
    if p is None:
        return None
    with p.open(encoding="utf-8") as f:
        return json.load(f)

for title, filename in [
    ("Original fully connected training", "EMNIST_Balanced_baseline_Training_Summary.json"),
    ("Expanded fully connected training", "EMNIST_Balanced_expanded_Training_Summary.json"),
]:
    d = load_json(filename)
    if d is not None:
        print(f"\n{title}: {filename}")
        for key in ("architecture", "sizes", "best_epoch", "best_validation_epoch", "test_correct", "test_total", "test_accuracy"):
            if key in d:
                print(f"  {key}: {d[key]}")

### Historical saved parameters

The original and expanded dense experiments retained text-form weights-and-biases exports. The repository distinguishes **best** and **final** saved weights where both were produced. A final-epoch export is not automatically the best validation checkpoint.

The later FPGA CNN did **not** simply reuse these dense-network weights.

In [ ]:
dense_exports = [
    "EMNIST_Balanced_baseline_WeightsAndBiases.txt",
    "EMNIST_Balanced_expanded_Best_WeightsAndBiases.txt",
    "EMNIST_Balanced_expanded_Final_WeightsAndBiases.txt",
    "EMNIST_Balanced_MLP200_Best_WeightsAndBiases.txt",
    "EMNIST_Balanced_MLP200_Final_WeightsAndBiases.txt",
]
for filename in dense_exports:
    path = ROOT / filename
    print(f"{filename}: {'present' if path.is_file() else 'not present'}")

## 4. Train and evaluate the 784–200–47 dense model

The `train_emnist_mlp_200.py` experiment used **784 input values, 200 hidden units, and 47 output scores**. The hidden layer used sigmoid activation. The PyTorch training loss was multiclass softmax cross-entropy, and the optimizer was **Adam**, not AdamW.

Its saved JSON weight/bias exports supported comparison with the earlier `network2`-style models. These exports do **not**, by themselves, establish ZyNet or FPGA hardware compatibility.

### Dense-model training evidence

![MLP-200 training results](emnist_mlp_train_result.png)

In [ ]:
mlp = load_json("training_summary_mlp200.json")
if mlp:
    for key in ("architecture", "hidden_activation", "optimizer", "batch_size", "best_validation_epoch", "best_validation_correct", "best_validation_total", "test_correct", "test_total", "test_accuracy"):
        if key in mlp:
            print(f"{key}: {mlp[key]}")
    if mlp.get("test_correct") is not None and mlp.get("test_total"):
        print(f"Independent test accuracy: {mlp['test_correct']}/{mlp['test_total']} = {100*mlp['test_correct']/mlp['test_total']:.2f}%")

## 5. Initial CNN reference experiment

The first retained CNN experiment is represented by `train_emnist_cnn_reference.py`. It provides a software convolutional reference for comparison with the larger improved CNN and the compact deployment candidate.

Its checkpoint and JSON summary are retained separately. As in the MNIST model notebook, **saved-model evaluation and training-screen evidence are not treated as interchangeable**.

### Initial CNN training evidence

![Initial CNN training result](emnist_cnn_train_result.png)

In [ ]:
reference = load_json("training_summary.json")
if reference:
    print("Best validation epoch:", reference.get("best_validation_epoch"))
    print("Best validation correct:", reference.get("best_validation_correct"), "/", reference.get("best_validation_total"))
    print("Test correct:", reference.get("test_correct"), "/", reference.get("test_count", reference.get("official_test_count")))
    print("Test accuracy:", f"{100*reference['test_accuracy']:.2f}%" if 'test_accuracy' in reference else "not recorded")

## 6. Improved CNN experiment

The `train_emnist_cnn_improved.py` script implemented a larger CNN with four convolutional layers (32, 32, 64, and 64 channels), followed by a 128-unit dense hidden layer and 47 output scores. Its training used image augmentation and dropout, with **AdamW** optimization.

This experiment provided a higher-accuracy software comparison model, but the larger architecture was not the selected compact FPGA candidate.

### Improved CNN training evidence

![Improved CNN training results](emnist_cnn2_train_result.png)

In [ ]:
improved = load_json("training_summary_improved.json")
if improved:
    for key in ("architecture", "best_validation_epoch", "best_validation_correct", "best_validation_total", "test_correct", "test_accuracy"):
        if key in improved: print(f"{key}: {improved[key]}")
    if improved.get("test_accuracy") is not None:
        print(f"Test accuracy: {100*improved['test_accuracy']:.2f}%")

## 7. Select the compact CNN

The `train_emnist_compact_fpga_candidate.py` experiment reduced model complexity while preserving convolutional feature extraction.

```text
28 × 28 grayscale input
    ↓
Conv2d 1 → 16, 3 × 3 + ReLU
    ↓
2 × 2 max pooling
    ↓
Conv2d 16 → 32, 3 × 3 + ReLU
    ↓
2 × 2 max pooling
    ↓
Conv2d 32 → 64, 3 × 3 + ReLU
    ↓
Global average pooling
    ↓
Dense 64 → 47
    ↓
argmax prediction
```

This network contains **26,351 trainable parameters**. Training used the same **101,520 / 11,280** stratified training/validation split as the CNN comparisons. The independent official test set contained **18,800 images**.

The compact model used **AdamW** with a default learning rate of 0.001, weight decay of 0.0005, mini-batches of 128, and validation-based checkpoint selection. The script allowed up to 40 epochs with early stopping.

### Compact CNN training evidence

![Compact CNN training results](emnist_compact_train_result.png)

The screenshot documents the training run. The JSON summary and checkpoint provide the authoritative selected-model record.

In [ ]:
compact = load_json("training_summary_compact.json")
if compact:
    for key in ("architecture", "parameter_count", "training_count", "validation_count", "official_test_count", "best_validation_epoch", "best_validation_correct", "best_validation_total", "best_validation_accuracy", "test_correct", "test_accuracy"):
        if key in compact: print(f"{key}: {compact[key]}")
    if compact.get("test_correct") is not None and compact.get("official_test_count"):
        print(f"Test accuracy: {compact['test_correct']}/{compact['official_test_count']} = {100*compact['test_correct']/compact['official_test_count']:.2f}%")

### Validation histories

The training summaries retain epoch-level values. The CSV files also provide a record of validation accuracy and training loss without requiring the original 28 × 28 image dataset to be committed.

The plot below is **reconstructed from the retained CSV evidence**, not from a newly trained model.

In [ ]:
def read_history(filename):
    p = ROOT / filename
    if not p.is_file():
        print("History unavailable:", filename)
        return []
    with p.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

history_files = {
    "Reference CNN": "validation_history.csv",
    "Improved CNN": "validation_history_improved.csv",
    "Compact CNN": "validation_history_compact.csv",
}
fig, ax = plt.subplots(figsize=(9, 4.5))
plotted = 0
for label, filename in history_files.items():
    rows = read_history(filename)
    if not rows or not {"epoch", "val_accuracy"}.issubset(rows[0]):
        print(f"Skipping {filename}: expected epoch and val_accuracy fields")
        continue
    ax.plot([int(r["epoch"]) for r in rows], [float(r["val_accuracy"])*100 for r in rows], label=label)
    plotted += 1
if plotted:
    ax.set(xlabel="Training epoch", ylabel="Validation accuracy (%)", title="CNN validation accuracy across training")
    ax.grid(alpha=.3); ax.legend(); plt.show()
else:
    plt.close(fig)

## 8. Compare reference-model test accuracy

All three CNN summaries report classifications on the **same official 18,800-image test partition**. Comparisons below use the retained results. They do not involve rerunning training or performing FPGA inference.

In [ ]:
cnn_results = []
for name, d in [("Reference CNN",reference),("Improved CNN",improved),("Compact CNN",compact)]:
    if d and d.get("test_correct") is not None:
        total = d.get("test_count", d.get("official_test_count", 18800))
        cnn_results.append((name, int(d["test_correct"]), int(total), 100*int(d["test_correct"])/int(total)))
print(f"{'Model':<20} {'Correct':>8} {'Total':>8} {'Accuracy':>10}")
for name, correct, total, accuracy in cnn_results:
    print(f"{name:<20} {correct:>8} {total:>8} {accuracy:>9.2f}%")

### Test-set confusion matrices

The available confusion-matrix CSVs support inspection of classification patterns for the CNN reference and compact models. Unlike aggregate accuracy, a confusion matrix identifies which character classes were confused, although it does not by itself explain why misclassifications occurred.

In [ ]:
for label, filename in [("Reference CNN", "test_confusion_matrix.csv"), ("Compact CNN", "test_confusion_matrix_compact.csv")]:
    p = ROOT / filename
    if not p.is_file():
        print("Missing:", filename)
        continue
    matrix = np.loadtxt(p, delimiter=",")
    print(label, "shape:", matrix.shape, "total:", int(matrix.sum()), "correct:", int(np.trace(matrix)))
    if matrix.shape == (47,47):
        fig, ax = plt.subplots(figsize=(7,6))
        img=ax.imshow(matrix, interpolation='nearest', cmap='Greys')
        ax.set(xlabel="Predicted class",ylabel="Actual class",title=label+" test confusion matrix")
        fig.colorbar(img,ax=ax);plt.show()

## 9. Saved-model artifacts and checkpoint provenance

The experiments retained PyTorch checkpoint files and, for selected dense networks, serialized weights-and-biases exports. The relevant checkpoint names in the GitHub directory are:

| File | Role |
|---|---|
| `best_cnn_checkpoint.pt` | Reference CNN checkpoint |
| `best_cnn_checkpoint_improved.pt` | Improved CNN checkpoint |
| `best_cnn_checkpoint_compact.pt` | **Selected compact CNN checkpoint** |
| `best_mlp_checkpoint.pt` | Best dense-model checkpoint |
| `final_mlp_checkpoint.pt` | Final dense-model checkpoint |

**Do not substitute one CNN checkpoint for another.** The architectures have different layer dimensions and cannot be interchanged. The selected compact checkpoint is the reference source of learned parameters for downstream deployments.

The following cell inspects *checkpoint metadata* without altering the files. It requires PyTorch; it does not require the original dataset.

In [ ]:
try:
    import torch
except ImportError:
    print("PyTorch not installed; checkpoint inspection skipped.")
else:
    for filename in ("best_cnn_checkpoint.pt", "best_cnn_checkpoint_improved.pt", "best_cnn_checkpoint_compact.pt", "best_mlp_checkpoint.pt", "final_mlp_checkpoint.pt"):
        p = ROOT / filename
        if not p.is_file():
            print(filename, "not present")
            continue
        try:
            obj = torch.load(p, map_location="cpu", weights_only=True)
            print("\n", filename)
            if isinstance(obj, dict):
                print("Top-level fields:", list(obj)[:20])
                weights = obj.get("model_state_dict")
                if isinstance(weights, dict):
                    print("Parameter count:", sum(v.numel() for v in weights.values()))
                    print("First layers:", [(k,tuple(v.shape)) for k,v in list(weights.items())[:3]])
        except Exception as exc:
            print(filename, "could not be inspected:", exc)

## 10. Preliminary quantization comparison

The retained `compare_emnist_cnn_quantization.py` software comparison tested lower-precision rounding of CNN weights and activations to evaluate suitability for implementation. **It was not FPGA inference**: floating-point accumulation and bias handling remained in the software comparison.

The corresponding `quantization_comparison.csv` contains the retained results, and the screenshot provides evidence of the comparison run.

![EMNIST CNN software quantization comparison](emnist_quantization_comparison.png)

In [ ]:
quant_path = artifact("quantization_comparison.csv")
if quant_path:
    with quant_path.open(newline="", encoding="utf-8-sig") as f:
        rows=list(csv.DictReader(f))
    print("Rows:",len(rows))
    if rows:
        print("Columns:",list(rows[0]))
        for row in rows[:15]: print(row)

## 11. Reproducibility boundaries

The model-generation scripts and saved checkpoints document the **software training and model-selection stage**. The later deployment implementations use the selected compact CNN as their reference, but subsequent conversions must be validated independently.

This notebook does **not** assert that:

- software fake-quantization results are bit-exact FPGA results;
- a checkpoint alone establishes RTL correctness;
- the original dataset is hosted in the GitHub branch;
- training screenshots replace the saved validation/test summaries; or
- a single selected checkpoint evaluation represents repeated independent training trials.

For the actual model instance used in later implementation, use the saved compact checkpoint rather than retraining and assuming identical parameter values. The reproducibility scripts preserve deterministic split seeds, but retraining on a different software environment may still produce differing weights or accuracy.

## 12. Final model artifacts

| Artifact | Role |
|---|---|
| `EMNIST_Balanced_Verification_Summary.txt` | Official dataset structure check |
| `EMNIST_Balanced_Orientation_Check.png` | Evidence of required upright-image correction |
| `EMNIST_Balanced_Reference_Vectors.npz` | One canonical, upright test vector per class |
| Dense training summaries and weight/bias exports | Historical fully connected experiments |
| `training_summary.json` | CNN software reference result |
| `training_summary_improved.json` | Improved CNN model result |
| `training_summary_compact.json` | Selected compact CNN result |
| `best_cnn_checkpoint_compact.pt` | **Selected model weights for downstream implementation** |
| Validation-history and confusion-matrix CSV files | Software training and test evidence |
| `quantization_comparison.csv` | Preliminary precision feasibility comparison |

### Final validated software-model result

For the selected compact CNN, the retained training summary records:

```text
16,327 / 18,800 = 86.85% test accuracy
```

This is the **software reference-model result**, not a measured FPGA classification result. The improved CNN achieved higher test accuracy, but the compact model reduced the computational and storage requirements relevant to the planned embedded implementation.

## Reproducibility note

The source training scripts, saved summaries, and checkpoints jointly establish how the selected model was produced. The notebook documents the historical progression rather than reconstructing each experiment as though it were performed again.

To reproduce training from scratch, obtain the **EMNIST Balanced** dataset from NIST and follow the scripts and dependencies in `EMNIST-Models-Code`. Re-execution may yield slightly different results depending on runtime and numerical libraries.

The retained checkpoint preserves the particular trained instance selected for the dissertation. Integer arithmetic adaptation, RTL verification, hardware resource measurements, power measurements, and Unity execution belong with their respective implementation materials.